In [2]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

def backtest(average_scores_df, stock):
    average_scores_df = average_scores_df[['datetime', 'instrument', 'score']]
    average_scores_df['datetime'] = average_scores_df['datetime'].astype('datetime64[ns]')
    average_scores_df = average_scores_df.sort_values(by=['datetime', 'instrument'], ascending=[True, True])
    average_scores_df = average_scores_df[average_scores_df['instrument'].isin(stock)]

    stock_df = pd.read_csv('/home/liyuante/nerocomputing24/dataset/zz500_2018_2023_new_2.csv')
    stock_df['dt'] = pd.to_datetime(stock_df['dt'])
    stock_df = stock_df[['kdcode', 'dt', 'close']]
    stock_df = stock_df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'})
    stock_df = stock_df.sort_values(by=['instrument', 'datetime'])
    stock_df['close_r'] = stock_df['close'] / stock_df['close'].shift(1)
    df_trade = stock_df[stock_df['datetime'] > '2022-12-31']

    x = pd.merge(df_trade, average_scores_df, on=['datetime', 'instrument'], how='outer')
    trade_date_unique = df_trade['datetime'].unique()
    df_return = pd.DataFrame()

    for date in trade_date_unique:
        x_day = x[x['datetime'] == date]
        r_day = x_day.nlargest(10, columns='score').close_r.mean()
        r_day = r_day - 1
        b = {'datetime': date, 'daily_return': r_day}
        df_return = df_return.append(b, ignore_index=True)

    portfolio_df_performance = df_return.set_index(['datetime'])

    alpha_df_performance = pd.DataFrame()
    alpha_df_performance['portfolio_daily_return'] = portfolio_df_performance['daily_return']
    alpha_df_performance['portfolio_net_value'] = (alpha_df_performance['portfolio_daily_return'] + 1).cumprod()

    net_value_columns = ['portfolio_net_value']

    alpha_statistics_df = pd.DataFrame(index=alpha_df_performance[net_value_columns].columns,
                                        columns=["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"])

    alpha_df_performance.index = pd.to_datetime(alpha_df_performance.index)
    alpha_statistics_df.loc[:, "年化收益"] = np.mean((alpha_df_performance[net_value_columns].tail(1)) ** (252 / len(alpha_df_performance)) - 1)
    alpha_statistics_df.loc[:, "年化波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252)
    alpha_statistics_df.loc[:, "最大回撤率"] = np.min((alpha_df_performance[net_value_columns] - alpha_df_performance[net_value_columns].cummax()) / alpha_df_performance[net_value_columns].cummax())
    alpha_statistics_df.loc[:, "夏普率"] = alpha_statistics_df["年化收益"] / alpha_statistics_df["年化波动率"]
    alpha_statistics_df.loc[:, "Calmar"] = alpha_statistics_df["年化收益"] / abs(alpha_statistics_df["最大回撤率"])
    alpha_statistics_df.loc[:, "IR"] = np.mean(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252) / np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)

    return alpha_statistics_df

def get_merged_score_df(base_path, exp_folder_index):
    # 创建一个字典，以存储每个日期文件的所有数据框
    data_frames_dict = {}

    # 遍历每个 prediction 文件夹
    for prediction_index in range(20):  # 假设有20个 prediction 文件夹
        current_folder_path = os.path.join(base_path, f'prediction_{prediction_index}', str(exp_folder_index))
        
        # 遍历当前文件夹中的所有 CSV 文件
        for file_name in os.listdir(current_folder_path):
            file_path = os.path.join(current_folder_path, file_name)
            if file_name.endswith('.csv') and os.path.isfile(file_path):
                df = pd.read_csv(file_path)
                if file_name in data_frames_dict:
                    data_frames_dict[file_name].append(df)
                else:
                    data_frames_dict[file_name] = [df]

    # 对每个日期文件的所有 DataFrame 进行合并并计算均值
    averaged_dfs = []
    for file_name, dfs in data_frames_dict.items():
        merged_df = pd.concat(dfs)
        averaged_df = merged_df.groupby(['kdcode', 'dt']).mean().reset_index()
        averaged_dfs.append(averaged_df)

    if averaged_dfs:
        # 合并所有日期的平均数据框
        return pd.concat(averaged_dfs)
    else:
        return pd.DataFrame()  # 如果没有文件，返回一个空的 DataFrame

base_path = '/home/liyuante/20240713_csi500_134'
d = pd.read_csv('/home/liyuante/nerocomputing24/dataset/zz500_2018_2023_new_2.csv')
stock = d['kdcode'].unique()

results = []

# 20次实验，每次实验包含0轮次，1轮次，...，5轮次
for exp_folder_index in tqdm(range(5)):
    # 得到0轮次，20次实验的平均值
    average_scores_df = get_merged_score_df(base_path, exp_folder_index)
    average_scores_df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'}, inplace=True)
    # print(average_scores_df)
    result = backtest(average_scores_df, stock)
    result = result[["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"]]  # 只保留6个指标
    # result['folder'] = exp_folder_index
    results.append(result)

final_df = pd.concat(results).reset_index(drop=True)
print(final_df)


100%|██████████| 5/5 [02:22<00:00, 28.45s/it]

       年化收益     年化波动率     最大回撤率       夏普率    Calmar        IR
0  0.007309  0.166501 -0.229848  0.043897  0.031798  0.024342
1 -0.245052  0.172834 -0.323720 -1.417844 -0.756988 -1.653492
2  0.006725  0.158075 -0.238723  0.042542  0.028170 -0.009435
3 -0.054195  0.164907 -0.155426 -0.328639 -0.348686 -0.388345
4 -0.154412  0.168718 -0.248581 -0.915204 -0.621173 -0.997938


In [2]:
final_df

,年化收益,年化波动率,最大回撤率,夏普率,Calmar,IR,folder
0,0.490968,0.206509,-0.136328,2.377463,3.601368,2.003995,0
1,0.366611,0.220917,-0.108049,1.659495,3.393012,1.490871,1
2,0.272061,0.196657,-0.105733,1.383430,2.573083,1.423053,2
3,0.436592,0.201203,-0.083286,2.169911,5.242065,2.005087,3
4,0.279814,0.215489,-0.169037,1.298504,1.655342,1.347084,4
5,0.198576,0.223120,-0.168900,0.889996,1.175698,1.006126,5
6,0.160267,0.198278,-0.111571,0.808298,1.436456,0.948932,6
7,0.163988,0.202271,-0.110098,0.810735,1.489466,0.942290,7
8,0.559737,0.215636,-0.104841,2.595742,5.338918,2.138518,8
9,0.610354,0.218321,-0.103341,2.795676,5.906200,2.334862,9
